# Multiline TRL

Multiline TRL 是一种使用至少两条不同物理长度的传输线和至少一个在两个端口上相同的反射标准进行校准的双端口 VNA 校准。线路的电气参数不需要知道，但传输线路应具有相同的结构（相同的传播常数和特性阻抗）。反射标准的反射系数不需要精确知道，相位需要以 90 度精度知道。

如果线路的测量相位差是 180 度的倍数，则校准是奇异的。校准准确度随着线路测量相位接近奇异性而变差，在两条线路情况下相位差为 90 度时获得最佳准确度。可以使用多条线路来扩展校准准确的频率范围。

此示例演示如何使用 `skrf` 的 NIST 风格 Multiline 校准。首先介绍一个 [简单应用](#Simple-Multiline-TRL)，然后进行 [完整模拟](#Compare-calibrations-with-different-combinations-of-lines) 以演示与线路数量相关的校准准确度改进。所有演示中使用的数据都由 skrf 生成，此代码在[示例末尾](#Simulation-to-generate-the-input-data)给出。

此外，此示例展示了 TUG multiline 校准（TUGMultilineTRL），它是常用 NIST 风格 multiline 方法的一种替代方案。TUG 方法提供更好的统计特性。还提供了一个示例通过 [Monte Carlo 分析](#monte-carlo-analysis---nist-vs-tug)比较这两种校准方法。

## 简单 Multiline TRL

## 设置

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

import skrf
from skrf.calibration import NISTMultilineTRL, TUGMultilineTRL, terminate
from skrf.media import CPW, Coaxial
from skrf.network import two_port_reflect

skrf.stylely()

## 将数据加载到 skrf

In [ ]:
# Load all measurement data into a dictionary
data = skrf.io.general.read_all_networks('multiline_trl_data/')

# Pull out measurements by name into an ordered list
measured_names = ['thru','reflect','linep3mm','line2p3mm']
measured = [data[k] for k in measured_names]

# Switch terms
gamma_f,gamma_r = data['gamma_f'],data['gamma_r']

# DUT
dut_meas = data['DUT']

# 50 ohm termination
res_50ohm_meas = data['res_50ohm']

## 简单 Multiline TRL

In [ ]:
# define the line lengths in meters (including thru)
l = [0, 0.3e-3, 2.3e-3]

# Do the calibration
cal = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r) # Switch terms
    )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)

corrected.plot_s_db()

## 比较不同线路组合的校准

这里我们遍历不同的线路组合以演示校准准确度的差异。

In [ ]:
# Run NIST Multiline TRL calibration with different combinations of lines

# Put through and reflect to their own list ...
mtr = measured[:2]

# and lines on their own
mlines = measured[2:]

line_len = l[1:]

cals = []
duts = []

line_combinations = [[0], [1], [0,1]]

for used_lines in line_combinations:

    m = mtr + [mlines[i] for i in used_lines]

    # Add thru length to list of line lengths
    l = [l[0]] + [line_len[i] for i in used_lines]

    # Do the calibration
    cal = NISTMultilineTRL(
        measured = m,  # Measured standards
        Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
        l = l,         # Lengths of the lines
        er_est = 7,    # Estimate of transmission line effective permittivity
        switch_terms = (gamma_f, gamma_r) # Switch terms
        )

    # Correct the DUT using the above calibration
    corrected = cal.apply_cal(dut_meas)
    corrected.name = f'DUT, lines {used_lines}'

    duts.append(corrected)
    cals.append(cal)

### 已校正 DUT 的传输

绘制使用不同校准线路组合校正的 DUT 的传输。

In [ ]:
plt.figure()
plt.title('DUT S21')
for dut in duts:
    dut.plot_s_db(m=1, n=0)

### 使用不同数量校准线路的校正 DUT S11

S11 显示更大的变化。

* 使用一条短线路时，低频非常嘈杂
* 仅使用长线路时，在校准准确度较差的频率处，thru 和线路的相位差接近 180 度的倍数
* 使用两条线路时，校准准确度在所有地方都很好

In [ ]:
plt.figure()
plt.title('DUT S11')
for dut in duts:
    dut.plot_s_db(m=0, n=0)

### 不同校准的归一化标准差

归一化标准差可用于测量校准的准确度。较小的数值表示校准对测量噪声不那么敏感。

 * 使用一条 90 度长线路的 TRL 校准的归一化标准差为 1。
 * 使用一条 180 度长无损线路的 TRL 校准是奇异的，具有无限的归一化标准差。
 * 使用多条线路时，可能获得小于 1 的归一化标准差。

请注意，nstd 已归一化，使其不考虑实际的测量噪声。它仅根据求解的传播常数和线路长度计算。它可以有多大取决于被测设备、测量噪声和测量的所需准确度。如果有大的尖峰，例如在下图的长线路情况下可见，这表明校准在该频率处非常接近奇异，测量准确度会很差。

In [ ]:
f_ghz = dut.frequency.f_scaled

plt.figure()
plt.title('Calibration normalized standard deviation')
for e, cal in enumerate(cals):
    plt.plot(f_ghz, cal.nstd, label=f'Lines: {line_combinations[e]}')
plt.ylim([0,20])
plt.legend(loc='upper right')
dut.frequency.labelXAxis()

## 计算校准中使用的传输线路的有效复相对介电常数

传输线路的有效复相对介电常数 $\epsilon_{r,eff}$ 与传播常数 $\gamma$ 的关系如下：

$\gamma = \frac{2\pi f}{c}\sqrt{\epsilon_{r,eff}}$，其中 $c$ 等于光速且 $f$ 是频率。

通常它是一个复数值，虚部表示损耗。

In [ ]:
# Define calibration standard media
freq = dut.frequency

# Get the cal with the both lines
cal = cals[-1]

plt.figure()
plt.title('CPW effective permittivity (real part)')
plt.plot(f_ghz, cal.er_eff.real, label='Solved er_eff')
plt.xlabel('Frequency (GHz)')
plt.legend(loc='lower right')

当线路长度差为 90 度时，TRL 校准准确度最好。然而，求解的传播常数和有效介电常数随着线路长度差的增大而更准确。在低频处，由于线路相位差较小，估计更嘈杂。

## 绘制求解的反射系数的相位

将校准应用于测量的反射标准，我们可以获得未知反射的已校正 S 参数。

In [ ]:
plt.figure()
plt.title('Solved reflection coefficient of the reflect standard')
cal.apply_cal(measured[1]).plot_s_deg(n=0, m=0, label='Solved short')

## 参考平面移动

由于在校准期间求解了介质的传播常数，可以通过指定距离移动参考平面。

参考平面移动可以用 `ref_plane` 参数指定。移动应指定为米，负长度朝向 VNA。默认情况下，对两个端口应用相同的移动。通过传递一个两个元素的列表支持两个端口上的不等移动。

In [ ]:
cal_shift = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r), # Switch terms
    # Shift reference planes twords VNA by this amount (in m) on both ports
    ref_plane = -50e-6
    )

# Correct the DUT using the above calibration
corrected_thru = cal.apply_cal(measured[0])
corrected_thru_shifted = cal_shift.apply_cal(measured[0])

corrected_thru.plot_s_deg(m=1, n=0, label='Thru phase')
corrected_thru_shifted.plot_s_deg(m=1, n=0, label='Reference plane shifted thru phase')

## 校准参考阻抗重新归一化

校准的参考阻抗默认为传输线路特性阻抗。如果我们知道线路的实际特性阻抗，可以通过 `z0_line` 参数将其提供给校准例程，将测量的 S 参数重新归一化为固定的参考 `z0_ref`。

如果单位长度的电导（G）远小于单位长度的电容性电抗（$j\omega C_0$），传输线路的特性阻抗可以用传播常数 $\gamma$ 和单位长度电容 $C_0$ 表示：

$Z_0 = \gamma/(j 2 \pi f C_0)$

如果 $C_0$ 已知，可以通过 `c0` 参数将其提供给校准例程，在假设 G = 0 的情况下将校准参考阻抗重新归一化为 `z0_ref`（默认为 50 欧姆）。
如果线路有损耗，特性阻抗是复数值，给出单个 `c0` 而不是固定的 `z0_line` 通常更准确。

在这种情况下，我们知道线路特性阻抗实际上是 55 欧姆。要从 55 欧姆重新归一化校准到 50 欧姆，我们需要将 `z0_line=55` 参数提供给校准例程。

In [ ]:
cal_ref = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r), # Switch terms
    z0_line = 55, # Line actual characteristic impedance
    z0_ref = 50 # Calibration reference impedance
    )

cal.apply_cal(res_50ohm_meas).s11.plot_s_db(label=r'50 $\Omega$ termination |$S_{11}$|, Z_ref = line')
cal_ref.apply_cal(res_50ohm_meas).s11.plot_s_db(label=r'50 $\Omega$ termination |$S_{11}$|, Z_ref = 50 $\Omega$')

重新归一化后，50 欧姆端接测量显示出良好的匹配。由于测量中的噪声，它并不完美匹配。

## TUG multiline TRL

TUG multiline TRL 是使用与 NIST 不同方法实现 multiline TRL 校准的方法。TUG multiline 中的校准标准定义与 NIST multiline 相同，要求至少两条线路和对称反射。关于 TUG multiline 的更多数学细节可以在以下找到：https://ziadhatab.github.io/posts/multiline-trl-calibration/

下面是比较 NIST multiline 和 TUG multiline 的示例。两种方法给出相同的结果。然而，在后面的 [Monte Carlo (MC) 分析](#monte-carlo-analysis---nist-vs-tug)中，我们将看到与 NIST 方法相比，TUG 方法提供更好的统计性能。

In [ ]:
# Load all measurement data into a dictionary
data = skrf.io.general.read_all_networks('multiline_trl_data/')
# Pull out measurements by name into an ordered list
measured_names = ['thru','reflect','linep3mm','line2p3mm']
measured = [data[k] for k in measured_names]
# Switch terms
gamma_f,gamma_r = data['gamma_f'],data['gamma_r']
# DUT
dut_meas = data['DUT']
# define the line lengths in meters (including thru)
l = [0, 0.3e-3, 2.3e-3]

# NIST multiline TRL
cal = NISTMultilineTRL(
    measured = measured,  # Measured standards
    Grefls = [-1], # Reflection coefficient of the reflect, -1 for short
    l = l,         # Lengths of the lines
    er_est = 7,    # Estimate of transmission line effective permittivity
    switch_terms = (gamma_f, gamma_r) # Switch terms
    )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'DUT NIST'

corrected.plot_s_db()

# TUG multiline TRL - note the difference in
cal = TUGMultilineTRL(
        line_meas=[measured[0]]+measured[2:],
        line_lengths=l,
        er_est=7,
        reflect_meas=[measured[1]],
        reflect_est=[-1],
        switch_terms = (gamma_f, gamma_r)
        )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'DUT TUG'

corrected.plot_s_db(linestyle='--')

## 无反射测量的校准

`TUGMultilineTRL` 的一个很好的功能是，如果你只关心传播常数和已校正 DUT 的 S21 和 S12，可以只使用线路标准。但是，要获得已校正 DUT 的正确 S11 和 S22 值，你还需要提供反射标准的测量。这可以在下面的示例中看到，当使用和不使用反射标准时，S11 和 S22 不重叠。

In [ ]:
# Full TUG multiline TRL
cal = TUGMultilineTRL(
        line_meas=[measured[0]]+measured[2:],
        line_lengths=l,
        er_est=7,
        reflect_meas=[measured[1]],
        reflect_est=[-1],
        switch_terms = (gamma_f, gamma_r)
        )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'TUG w/ reflect'

corrected.plot_s_db(linestyle='-')

# TUG multiline without reflect measurement
cal = TUGMultilineTRL(
        line_meas=[measured[0]]+measured[2:],
        line_lengths=l,
        er_est=7,
        switch_terms = (gamma_f, gamma_r)
        )

# Correct the DUT using the above calibration
corrected = cal.apply_cal(dut_meas)
corrected.name = 'TUG w/o reflect'

corrected.plot_s_db(linestyle='--')

## 校准特征值

NIST 方法采用归一化标准差来强调校准的敏感性。类似地，TUG 方法使用基于校准问题特征值的度量。TUG multiline 方法只求解一个加权特征分解，产生实数值特征值 $\lambda$。此特征值表示校准的稳定性。特征值越接近零，校准越敏感。本质上，它可以被视为 NIST 方法中定义的归一化标准差的相反值。

In [ ]:
f_ghz = dut.frequency.f_scaled

plt.figure()
plt.title('TUG multiline eigenvalue')
plt.plot(f_ghz, cal.lambd)
dut.frequency.labelXAxis()

## Monte Carlo 分析 - NIST vs TUG

使用 MC 分析，我们生成具有随机噪声的标准合成数据，并重复该过程多次，每次使用新的噪声样本。在每个试验中，我们还计算并存储已校正的 DUT，我们使用它来计算校准方法的平均绝对误差 (MAE)。MAE 的通用定义如下：

$$
\text{MAE} = \frac{1}{N}\sum_{i=1}^{N} |x_i - x_\mathrm{true}|
$$

其中 $x_i$ 是第 $i$ 个 MC 试验，$x_\mathrm{true}$ 是真实值。下面的代码生成参考真实值。

In [ ]:

freq = skrf.Frequency(1, 100, 201, 'GHz')
# CPW media used for DUT and the calibration standards
cpw = CPW(freq, w=40e-6, s=51e-6, ep_r=12.9, t=5e-6, rho=2e-8)
# 1.0 mm coaxial media for calibration error boxes
coax1mm = Coaxial(freq, z0_port=50, Dint=0.434e-3, Dout=1.0e-3, sigma=1e8)
f_ghz = cpw.frequency.f*1e-9
# Error boxes
X = coax1mm.line(1, 'm', z0=58, name='X')
Y = coax1mm.line(1.1, 'm', z0=40, name='Y')
# Realistic looking switch terms
gamma_f = coax1mm.delay_load(0.2, 21e-3, 'm', z0=60)
gamma_r = coax1mm.delay_load(0.25, 16e-3, 'm', z0=56)
# Lengths of the lines used in the calibration, units are in meters
line_lengths = [0, 0.3e-3, 2.3e-3]  # first is thru
lines = [cpw.line(l, 'm') for l in line_lengths]
# Attenuator with mismatched feed lines
dut_feed = cpw.line(1.5e-3, 'm', z0=60)
dut = dut_feed**cpw.attenuator(-10)**dut_feed
# Reflect standard
reflect_offset = 10e-6
short = cpw.delay_short(reflect_offset, 'm')
short = two_port_reflect(short, short)

# Measured... embedded in the error boxes
line_meas = [terminate(X**k**Y,gamma_f, gamma_r) for k in lines]
reflect_meas = [terminate(X**short**Y, gamma_f, gamma_r)]
dut_meas = terminate(X**dut**Y, gamma_f, gamma_r)
er_est = 6.3-0.0001j
reflect_est = [-1]

# NIST multiline TRL (Noiseless case as reference)
measured = [line_meas[0]] + [reflect_meas[0]] + line_meas[1:]
cal = NISTMultilineTRL(
    measured = measured,
    Grefls = reflect_est,
    l = line_lengths,
    refl_offset = reflect_offset,
    er_est = er_est,
    switch_terms=[gamma_f, gamma_r])

dut_ideal = cal.apply_cal(dut_meas)
dut_ideal.name = 'DUT'
dut_ideal.plot_s_db()

我们现在通过在极坐标中引入噪声来启动 MC 分析。

In [ ]:
# Monte Carlo Analysis
print('MC starts: ')
M = 100 # number of trials
mag_std = 0.05 # std of magnitude (linear)
phase_std = 20 # std of phase (degrees)

cal_dut_NIST = []
cal_dut_TUG  = []

for inx in range(M):
    # add noise (make sure to copy and not override the original data)
    lines_n   = [NW.copy() for NW in line_meas]
    for NW in lines_n:
        NW.add_noise_polar(mag_std, phase_std)

    reflect_n   = [NW.copy() for NW in reflect_meas]
    for NW in reflect_n:
        NW.add_noise_polar(mag_std, phase_std)

    # TUG multiline TRL
    cal = TUGMultilineTRL(line_meas=lines_n, line_lengths=line_lengths, er_est=er_est,
                reflect_meas=reflect_n, reflect_est=reflect_est, reflect_offset=reflect_offset,
                switch_terms=[gamma_f, gamma_r])
    cal_dut_TUG.append(cal.apply_cal(dut_meas))

    # NIST multiline TRL
    measured = [lines_n[0]] + [reflect_n[0]] + lines_n[1:]
    cal = NISTMultilineTRL(
        measured = measured,
        Grefls = reflect_est,
        l = line_lengths,
        refl_offset = reflect_offset,
        er_est = er_est,
        switch_terms=[gamma_f, gamma_r])
    cal_dut_NIST.append(cal.apply_cal(dut_meas))
    print(f'MC Index: {inx+1}/{M} done.', end='\r', flush=True)


最后，我们计算并绘制 MAE。结果显示，与 NIST 方法相比，TUG multiline 的误差更低。

In [ ]:
def dut_MAE(MC, ideal):
    # compute mean absolute error
    return np.array([ abs(x.s-ideal.s) for x in MC ]).mean(axis=0)

MAE_NIST = dut_MAE(cal_dut_NIST, dut_ideal)
MAE_TUG  = dut_MAE(cal_dut_TUG, dut_ideal)

f_ghz = dut_ideal.frequency.f_scaled

plt.figure()
plt.title('Mean Absolute Error')
plt.plot(f_ghz, MAE_NIST[:,0,0], label='NIST, S11')
plt.plot(f_ghz, MAE_NIST[:,1,0], label='NIST, S21')
plt.plot(f_ghz, MAE_TUG[:,0,0], label='TUG, S11')
plt.plot(f_ghz, MAE_TUG[:,1,0], label='TUG, S21')
plt.legend(loc='upper right')
dut_ideal.frequency.labelXAxis()

## 生成输入数据的模拟

这里是我们制作上面使用的数据的方法。

## 创建频率和介质

In [ ]:
freq = skrf.Frequency(1, 100, 201, 'GHz')

# CPW media used for DUT and the calibration standards
cpw = CPW(freq, w=40e-6, s=51e-6, ep_r=12.9,
                     t=5e-6, rho=2e-8)
print(cpw.z0[0])

# 1.0 mm coaxial media for calibration error boxes
coax1mm = Coaxial(freq, z0_port=50, Dint=0.434e-3, Dout=1.0e-3, sigma=1e8)
print(coax1mm.z0[0])

f_ghz = cpw.frequency.f*1e-9

## 创建逼真的误差网络

传播常数确定是迭代的，当误差网络随机生成时效果不好

In [ ]:
X = coax1mm.line(1, 'm', z0=58, name='X')
Y = coax1mm.line(1.1, 'm', z0=40, name='Y')

plt.figure()
plt.title('Error networks')
X.plot_s_db()
Y.plot_s_db()

# Realistic looking switch terms
gamma_f = coax1mm.delay_load(0.2, 21e-3, 'm', z0=60)
gamma_r = coax1mm.delay_load(0.25, 16e-3, 'm', z0=56)

plt.figure()
plt.title('Switch terms')
gamma_f.plot_s_db()
gamma_r.plot_s_db()

## 生成虚构测量

In [ ]:
# Lengths of the lines used in the calibration, units are in meters
line_len = [0.3e-3, 2.3e-3]
lines = [cpw.line(l, 'm') for l in line_len]

# Attenuator with mismatched feed lines
dut_feed = cpw.line(1.5e-3, 'm', z0=60)
dut = dut_feed**cpw.attenuator(-10)**dut_feed

res_50ohm = cpw.resistor(50) ** cpw.short(nports=2) ** cpw.resistor(50)

# Through and non-ideal short
# Real reflection coefficient is solved during the calibration

short = cpw.delay_short(10e-6, 'm')

actuals = [
    cpw.thru(),
    two_port_reflect(short, short),
    ]

actuals.extend(lines)

# Measured
measured = [X**k**Y for k in actuals]

# Switch termination
measured = [terminate(m, gamma_f, gamma_r) for m in measured]

# Add little noise to the measurements
for m in measured:
    m.add_noise_polar(0.001, 0.1)

names = ['thru', 'reflect', 'linep3mm', 'line2p3mm']
for k,name in enumerate(names):
    measured[k].name=name


# Noiseless DUT so that all the noise will be from the calibration
dut_meas = terminate(X**dut**Y, gamma_f, gamma_r)
dut_meas.name = 'DUT'

res_50ohm_meas = terminate(X**res_50ohm**Y, gamma_f, gamma_r)
res_50ohm_meas.name = 'res_50ohm'

# Put through and reflect to their own list ...
mtr = measured[:2]

# and lines on their own
mlines = measured[2:]

# write data to disk
write_data = False
if write_data:
    [k.write_touchstone(dir='multiline_trl_data/') for k in measured]
    gamma_f.write_touchstone('multiline_trl_data/gamma_f.s1p')
    gamma_r.write_touchstone('multiline_trl_data/gamma_r.s1p')
    dut_meas.write_touchstone(dir='multiline_trl_data/')
    res_50ohm_meas.write_touchstone(dir='multiline_trl_data/')